# 第 10 讲实验：Inference 与 KV Cache

本实验使用合成请求长度，估算 KV payload、contiguous padding 与 paged block allocation。它不是 vLLM 实测，也不会生成虚假的 TTFT/TBT 或 tokens/s。

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt

START = Path.cwd().resolve()
REPO_ROOT = next((p for p in (START, *START.parents) if (p / "code" / "lecture_10").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("请在课程仓库内启动 Jupyter Lab")
sys.path.insert(0, str(REPO_ROOT / "code" / "lecture_10"))
from kv_cache_lab import (
    bytes_to_gib, kv_cache_bytes_per_token, kv_cache_memory_report,
    max_tokens_for_kv_budget, summarize_request_trace,
)

requests = [
    {"prompt_tokens": 2049, "output_tokens": 128},
    {"prompt_tokens": 4097, "output_tokens": 256},
    {"prompt_tokens": 8193, "output_tokens": 64},
    {"prompt_tokens": 1025, "output_tokens": 192},
    {"prompt_tokens": 6145, "output_tokens": 96},
]
config = {"num_layers": 32, "num_kv_heads": 8, "head_dim": 128, "block_size": 16}
trace = summarize_request_trace(requests, bits_per_element=16, **config)
print("Synthetic requests:", trace["request_count"])
print("Prompt tokens:", trace["total_prompt_tokens"], "| output tokens:", trace["total_output_tokens"])
print("Final context lengths:", trace["final_sequence_lengths"])
print("KV bytes/token:", f"{trace['cache_memory']['bytes_per_token']:,}")
print("Payload-only estimates; not a model-serving benchmark.")

## 实验 1：Logical KV payload 与内存分配浪费

Logical payload 是实际上下文对应的 K/V 元素；contiguous batch 可能为短请求按最长序列 padding；paged allocation 按每条序列独立向 block size 对齐。

In [ ]:
cache = trace["cache_memory"]
for key in ("logical_bytes", "contiguous_allocated_bytes", "paged_allocated_bytes",
            "contiguous_waste_bytes", "paged_waste_bytes"):
    print(f"{key}: {bytes_to_gib(cache[key]):.4f} GiB")
print("page count:", cache["page_count"], "| block size:", cache["block_size"])

labels = ["logical payload", "contiguous padded", "paged blocks"]
values = [
    bytes_to_gib(cache["logical_bytes"]),
    bytes_to_gib(cache["contiguous_allocated_bytes"]),
    bytes_to_gib(cache["paged_allocated_bytes"]),
]
plt.figure(figsize=(7, 4))
plt.bar(labels, values)
plt.ylabel("KV payload/allocation (GiB)")
plt.title("Synthetic request trace, BF16/FP16 payload")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## 实验 2：KV 位宽与理论 payload

这里只按每个 K/V 元素的位宽估算 payload。真实量化缓存还可能需要 scale、zero-point、block metadata、对齐和专用 kernel 支持。

In [ ]:
bits_list = [16, 8, 4]
reports = {
    bits: kv_cache_memory_report(
        trace["final_sequence_lengths"], **{**config, "bits_per_element": bits}
    )
    for bits in bits_list
}
for bits, report in reports.items():
    print(
        f"{bits}-bit: {bytes_to_gib(report['logical_bytes']):.3f} GiB logical, "
        f"{bytes_to_gib(report['paged_allocated_bytes']):.3f} GiB paged payload"
    )
plt.figure(figsize=(6, 4))
plt.bar([f"{b}-bit" for b in bits_list],
        [bytes_to_gib(reports[b]["logical_bytes"]) for b in bits_list])
plt.ylabel("Logical KV payload (GiB)")
plt.title("Payload-only bit-width comparison")
plt.tight_layout()
plt.show()

## 实验 3：给定 KV budget 的 token 容量上界

预算必须是扣除模型权重、激活、临时 buffer、运行时开销之后真正留给 KV payload 的空间。这里的 4 GiB 是教学输入，不是设备可用显存测量。

In [ ]:
kv_budget = 4 * 1024**3
for bits in bits_list:
    tokens = max_tokens_for_kv_budget(
        kv_budget, num_layers=32, num_kv_heads=8, head_dim=128,
        bits_per_element=bits,
    )
    print(f"{bits}-bit payload: at most {tokens:,} cached tokens under 4 GiB")
print("Actual capacity is lower if metadata and non-KV allocations use part of the budget.")

## 实验 4：Paged block size 与内部碎片

Block 越大，block table 管理开销可能下降，但最后一个 block 的未使用 token slots 可能增加。这里仅计算分配粒度造成的内部浪费，不模拟 vLLM 的真实 block manager。

In [ ]:
block_sizes = [8, 16, 32, 64]
waste_gib = []
for block in block_sizes:
    report = kv_cache_memory_report(
        trace["final_sequence_lengths"], num_layers=32, num_kv_heads=8,
        head_dim=128, bits_per_element=16, block_size=block,
    )
    waste_gib.append(bytes_to_gib(report["paged_waste_bytes"]))
    print(f"block={block:2d}: pages={report['page_count']:4d}, "
          f"waste={bytes_to_gib(report['paged_waste_bytes']):.4f} GiB")
plt.figure(figsize=(6, 4))
plt.plot(block_sizes, waste_gib, "o-")
plt.xlabel("Block size (tokens)")
plt.ylabel("Internal fragmentation (GiB)")
plt.title("Paged allocation waste on synthetic trace")
plt.tight_layout()
plt.show()

## 复盘

1. KV cache bytes/token 中的因子 2、layers、KV heads 和 head_dim 分别代表什么？
2. 为什么 GQA/MQA 能减少 KV cache payload？
3. Paged allocation 减少了什么浪费，又不能消除什么成本？
4. 为什么 payload 位宽减半不保证端到端延迟减半？
5. 真实 serving benchmark 还需要报告哪些延迟、吞吐、质量与硬件指标？